# Phase 5 — One-Class Tree Detector + Two-Stage End-to-End Pipeline

## Why Phase 5 exists

Phase 4 proved the ResNet-18 crop classifier can distinguish Healthy vs Seca
(F1-Seca=0.354 on GT crops). But Stage 1 never had a properly trained detector:
- Phase 3 fine-tuned a 2-class model where classification gradients interfered with localisation
- Phase 4 Stage 1 fell back to the generic pretrained baseline (domain shift)

Phase 5 fixes this by training a **1-class 'Tree' detector** on all Dehesa annotations
(Healthy + Seca collapsed to 'Tree'). Simpler task, cleaner gradients, better localisation.
Then we combine it with the Phase 4 classifier for the **first true end-to-end evaluation**.

## Dataset note
Seca trees look identical to Healthy trees from above — same crown shape, similar colour.
The GT labels come from multi-temporal comparison (summer 2019 vs prior year), not visual inspection.
That's what makes this dataset scientifically valuable and the detection task genuinely hard.

In [ ]:
import platform, torch
print(f'Platform : {platform.system()} {platform.machine()}')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Cell 1: Imports & Config

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns
from omegaconf import OmegaConf

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── Paths (relative to repo root) ─────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT  = _cwd.parent if _cwd.name == 'notebooks' else _cwd
DATA_DIR   = REPO_ROOT / 'data'
MODELS_DIR = REPO_ROOT / 'models_v2'
REPORTS    = REPO_ROOT / 'reports'
for d in [DATA_DIR, MODELS_DIR, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

# Source data: CSVs from Phase 3 (03_fine_tuning.ipynb)
SRC_DATA  = DATA_DIR
TRAIN_CSV = SRC_DATA / 'train.csv'
VAL_CSV   = SRC_DATA / 'val.csv'
TEST_CSV  = SRC_DATA / 'test.csv'

# Phase 4 Stage 2 classifier (reused from 04_two_stage_pipeline.ipynb output)
STAGE2_PATH = MODELS_DIR / 'stage2_classifier.pt'
CROP_DIR    = DATA_DIR / 'crops'

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_EPOCHS   = 30
LR           = 1e-4
BATCH_SIZE   = 4 if DEVICE == 'cuda' else 1
NUM_WORKERS  = 4 if DEVICE == 'cuda' else 0
IOU_THRESH   = 0.4
SCORE_THRESH = 0.3
CROP_SIZE    = 96
PADDING      = 8

print(f'Repo root   : {REPO_ROOT}')
print(f'Device      : {DEVICE}')
print(f'Epochs      : {NUM_EPOCHS}  LR: {LR}  Batch: {BATCH_SIZE}')
print(f'Score thresh: {SCORE_THRESH}')
print(f'Stage2 path : {STAGE2_PATH} (exists: {STAGE2_PATH.exists()})')

## Cell 2: Build 1-Class CSVs

Collapse all labels (Healthy, Seca) to a single class 'Tree'.
This removes classification ambiguity from the detector — it only learns *where* trees are.

In [ ]:
def make_oneclass_csv(src_path, dst_path):
    df = pd.read_csv(src_path)
    df['label'] = 'Tree'
    df.to_csv(dst_path, index=False)
    return df

train_1c = make_oneclass_csv(TRAIN_CSV, DATA_DIR / 'train_1class.csv')
val_1c   = make_oneclass_csv(VAL_CSV,   DATA_DIR / 'val_1class.csv')
test_1c  = make_oneclass_csv(TEST_CSV,  DATA_DIR / 'test_1class.csv')

print('1-class CSVs created:')
for name, df in [('train', train_1c), ('val', val_1c), ('test', test_1c)]:
    print(f'  {name:<6}: {len(df):>5} annotations, {df["image_path"].nunique()} images, classes: {df["label"].unique()}')
print(f'\nTotal annotations used: {len(train_1c)+len(val_1c)+len(test_1c)}')
print('All labels collapsed to "Tree" — detector learns shape/texture, not class.')

## Cell 3: Train 1-Class DeepForest Detector

Same training setup as Phase 3 but with num_classes=1.
Fewer epochs needed (simpler task) and no classification head competition.

In [ ]:
from deepforest import main as df_main

print('Loading DeepForest 1-class model...')
detector = df_main.deepforest(config_args={
    'num_classes': 1,
    'val_accuracy_interval': 9999,
    'workers': NUM_WORKERS,
    'batch_size': BATCH_SIZE,
})
detector.load_model()
print(f'  num_classes : {detector.config.num_classes}')
print(f'  label_dict  : {detector.label_dict}')

# Configure training
# Images are in roboflow_dataset/{train,valid}/ (roboflow uses 'valid' not 'val')
OmegaConf.update(detector.config, 'train.csv_file',      str(DATA_DIR / 'train_1class.csv'),             merge=True)
OmegaConf.update(detector.config, 'train.root_dir',      str(SRC_DATA / 'roboflow_dataset' / 'train'),   merge=True)
OmegaConf.update(detector.config, 'validation.csv_file', str(DATA_DIR / 'val_1class.csv'),               merge=True)
OmegaConf.update(detector.config, 'validation.root_dir', str(SRC_DATA / 'roboflow_dataset' / 'valid'),   merge=True)
OmegaConf.update(detector.config, 'train.lr',            float(LR),             merge=True)
OmegaConf.update(detector.config, 'train.epochs',        int(NUM_EPOCHS),       merge=True)
OmegaConf.update(detector.config, 'batch_size',          int(BATCH_SIZE),       merge=True)
OmegaConf.update(detector.config, 'workers',             int(NUM_WORKERS),      merge=True)
# Use stepLR to avoid ReduceLROnPlateau requiring val_loss metric
OmegaConf.update(detector.config, 'train.scheduler.type', 'stepLR',             merge=True)
try:
    OmegaConf.update(detector.config, 'score_thresh', SCORE_THRESH, merge=True)
except Exception:
    pass

# CRITICAL: Trainer is built in __init__ before OmegaConf updates — must rebuild it
# so that max_epochs=NUM_EPOCHS and other settings take effect.
detector.create_trainer()
print(f'Trainer rebuilt — max_epochs: {detector.trainer.max_epochs}')

print(f'\n===== TRAINING: 1-class detector ({NUM_EPOCHS} epochs, LR={LR}) =====')
detector.trainer.fit(detector)
print('Training complete.')

## Cell 4: Save 1-Class Detector

In [ ]:
import torch as _torch
DETECTOR_PATH = MODELS_DIR / 'stage1_detector_1class.pt'
_torch.save(detector.state_dict(), DETECTOR_PATH)
size_mb = DETECTOR_PATH.stat().st_size / 1e6
print(f'Stage 1 detector saved: {DETECTOR_PATH}')
print(f'Size: {size_mb:.1f} MB')

## Cell 5: Load Phase 4 Stage 2 Classifier

We reuse the ResNet-18 classifier already trained in Phase 4.
No retraining needed — Stage 2 learned crop-level Healthy/Seca distinction independently.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
eval_tfm = T.Compose([
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
CLASS_NAMES = ['Healthy', 'Seca']

def build_classifier():
    m = models.resnet18(weights=None)
    m.fc = nn.Sequential(
        nn.Linear(m.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 2)
    )
    return m.to(DEVICE)

classifier = build_classifier()
classifier.load_state_dict(_torch.load(str(STAGE2_PATH), map_location=DEVICE))
classifier.eval()
print(f'Stage 2 classifier loaded from Phase 4: {STAGE2_PATH.name}')

## Cell 6: End-to-End Pipeline Evaluation

This is the first true end-to-end evaluation:
1. Stage 1 detects trees on full test tiles (no GT boxes used)
2. Each detected box is cropped and classified by Stage 2
3. Predictions are matched to GT boxes by IoU ≥ 0.4
4. F1 computed over matched pairs

This measures the *real* system performance, not just crop classification accuracy.

In [ ]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter == 0: return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)

def classify_crop(img, x1, y1, x2, y2):
    crop = img.crop((max(0,x1-PADDING), max(0,y1-PADDING),
                     min(img.width,x2+PADDING), min(img.height,y2+PADDING)))
    t = eval_tfm(crop).unsqueeze(0).to(DEVICE)
    with _torch.no_grad():
        logits = classifier(t)
        probs  = _torch.softmax(logits, dim=1)[0]
        pred   = logits.argmax(1).item()
    return CLASS_NAMES[pred], float(probs[pred])

test_df = pd.read_csv(TEST_CSV)
test_images = sorted(test_df['image_path'].apply(lambda p: Path(p).name).unique())

all_gt_labels, all_pred_labels = [], []
results_per_image = {}

print(f'Running end-to-end pipeline on {len(test_images)} test images...')
print(f'IoU threshold: {IOU_THRESH}  Score threshold: {SCORE_THRESH}')
print()

for img_name in test_images:
    candidates = list(SRC_DATA.rglob(img_name))
    if not candidates:
        print(f'  {img_name}: image file not found, skipping')
        continue
    img_path = candidates[0]

    # Stage 1 — detect trees
    try:
        preds_df = detector.predict_image(path=str(img_path))
    except Exception as e:
        print(f'  {img_name}: Stage1 error — {e}')
        continue
    if preds_df is None or len(preds_df) == 0:
        pred_boxes = []
    else:
        pred_boxes = [(int(r.xmin), int(r.ymin), int(r.xmax), int(r.ymax))
                      for _, r in preds_df.iterrows()]

    # Stage 2 — classify each detected crop
    img = Image.open(img_path).convert('RGB')
    pred_classified = []
    for (x1,y1,x2,y2) in pred_boxes:
        label, conf = classify_crop(img, x1, y1, x2, y2)
        pred_classified.append((x1,y1,x2,y2,label,conf))

    # Match predictions to GT boxes by IoU
    gt_rows = test_df[test_df['image_path'].str.contains(img_name, na=False)]
    gt_boxes = [(int(r.xmin),int(r.ymin),int(r.xmax),int(r.ymax),r.label)
                for _,r in gt_rows.iterrows()]

    matched_gt  = set()
    matched_pred = set()
    tp_pairs = []

    for pi, (px1,py1,px2,py2,plabel,pconf) in enumerate(pred_classified):
        best_iou, best_gi = 0.0, -1
        for gi, (gx1,gy1,gx2,gy2,glabel) in enumerate(gt_boxes):
            if gi in matched_gt: continue
            iou = compute_iou((px1,py1,px2,py2),(gx1,gy1,gx2,gy2))
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_iou >= IOU_THRESH and best_gi >= 0:
            matched_gt.add(best_gi)
            matched_pred.add(pi)
            gt_label = gt_boxes[best_gi][4]
            all_gt_labels.append(gt_label)
            all_pred_labels.append(plabel)
            tp_pairs.append((pred_classified[pi], gt_boxes[best_gi], best_iou))

    # Unmatched GT = false negatives
    for gi, (gx1,gy1,gx2,gy2,glabel) in enumerate(gt_boxes):
        if gi not in matched_gt:
            all_gt_labels.append(glabel)
            all_pred_labels.append('Background')  # missed

    h = sum(1 for _,_,l,_ in [(r[0],r[1],r[2],r[3]) for r in pred_classified] if l=='Healthy') if pred_classified else 0
    s = sum(1 for r in pred_classified if r[4]=='Seca')
    h = sum(1 for r in pred_classified if r[4]=='Healthy')
    print(f'  {img_name}: {len(pred_boxes)} detected, {len(gt_boxes)} GT, '
          f'{len(tp_pairs)} matched | pred: {h}H {s}S')
    results_per_image[img_name] = {
        'predictions': pred_classified, 'gt': gt_boxes, 'tp_pairs': tp_pairs
    }

print()
print('=== END-TO-END EVALUATION ===')
uniq_labels = ['Healthy', 'Seca', 'Background']
print(classification_report(all_gt_labels, all_pred_labels, labels=['Healthy','Seca'], zero_division=0))

f1_seca    = f1_score(all_gt_labels, all_pred_labels, labels=['Seca'],    average='macro', zero_division=0)
f1_healthy = f1_score(all_gt_labels, all_pred_labels, labels=['Healthy'], average='macro', zero_division=0)
f1_overall = f1_score(all_gt_labels, all_pred_labels, labels=['Healthy','Seca'], average='weighted', zero_division=0)
print(f'F1 Overall (H+S) : {f1_overall:.4f}')
print(f'F1 Healthy       : {f1_healthy:.4f}')
print(f'F1 Seca (KEY)    : {f1_seca:.4f}')

## Cell 7: Visualise End-to-End Results

In [ ]:
def visualise_e2e(img_path, img_name, data):
    img_arr = np.array(Image.open(img_path).convert('RGB'))
    fig, axes = plt.subplots(1, 1, figsize=(11, 9))
    ax = axes
    ax.imshow(img_arr)
    colours = {'Healthy': '#00cc44', 'Seca': '#ff3333', 'Tree': '#ffaa00'}
    for (x1,y1,x2,y2,label,conf) in data['predictions']:
        c = colours.get(label, 'orange')
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
            linewidth=2,edgecolor=c,facecolor='none'))
        ax.text(x1,max(0,y1-3),f'{label[:1]}{conf:.2f}',color=c,fontsize=6,
            fontweight='bold',bbox=dict(facecolor='black',alpha=0.5,pad=1,edgecolor='none'))
    for (gx1,gy1,gx2,gy2,glabel) in data['gt']:
        gc = '#00cc44' if glabel=='Healthy' else '#ff3333'
        ax.add_patch(patches.Rectangle((gx1,gy1),gx2-gx1,gy2-gy1,
            linewidth=1.5,edgecolor='white',facecolor='none',linestyle='--'))
    h = sum(1 for r in data['predictions'] if r[4]=='Healthy')
    s = sum(1 for r in data['predictions'] if r[4]=='Seca')
    gh = sum(1 for r in data['gt'] if r[4]=='Healthy')
    gs = sum(1 for r in data['gt'] if r[4]=='Seca')
    ax.set_title(f'{img_name}\nPredicted: {h} Healthy (green), {s} Seca (red) | '
                 f'GT: {gh} Healthy, {gs} Seca (white dashed)', fontsize=9)
    ax.axis('off')
    return fig

for i, (img_name, data) in enumerate(list(results_per_image.items())[:3], 1):
    candidates = list(SRC_DATA.rglob(img_name))
    if not candidates: continue
    fig = visualise_e2e(candidates[0], img_name, data)
    out = REPORTS / f'e2e_result_{i}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## Cell 8: Full Comparison — All Phases

In [ ]:
config = {
    'architecture': 'Two-Stage end-to-end: DeepForest 1-class + ResNet-18 classifier',
    'stage1': 'DeepForest fine-tuned 1-class Tree detector (Phase 5)',
    'stage2': 'ResNet-18 crop classifier (Phase 4, reused)',
    'stage1_epochs': NUM_EPOCHS,
    'stage1_lr': LR,
    'iou_threshold': IOU_THRESH,
    'score_thresh': SCORE_THRESH,
    'device': DEVICE,
    'seed': SEED,
    'f1_seca_e2e': float(f1_seca),
    'f1_healthy_e2e': float(f1_healthy),
    'f1_overall_e2e': float(f1_overall),
    'phase3_f1_seca': 0.0,
    'phase4_f1_seca_crops': 0.354,
    'improvement_vs_phase3': float(f1_seca),
    'improvement_vs_phase4': float(f1_seca - 0.354),
}
with open(MODELS_DIR / 'e2e_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('╔══════════════════════════════════════════════════════════════╗')
print('║      PHASE 5 — FULL PIPELINE COMPARISON                     ║')
print('╚══════════════════════════════════════════════════════════════╝')
print(f'{"Phase":<35} {"F1-Seca":>10} {"F1-Overall":>12}')
print('-' * 60)
print(f'{"Phase 2 — Zero-Shot baseline":<35} {0.000:>10.4f} {0.320:>12.4f}')
print(f'{"Phase 3 — 2-class fine-tune":<35} {0.000:>10.4f} {0.669:>12.4f}')
print(f'{"Phase 4 — 2-stage (GT crops)":<35} {0.354:>10.4f} {0.712:>12.4f}')
print(f'{"Phase 5 — 2-stage end-to-end":<35} {f1_seca:>10.4f} {f1_overall:>12.4f}')
print('=' * 60)
print()
print(f'Saved: models/e2e_config.json')